In [ ]:
import pandas as pd
import numpy as np

def add_home_away_features(df):
    df = df.copy()

    # Target: did home team win?
    # You can swap to homePoints/awayPoints if you prefer the CFBD scores
    df['home_win'] = (df['score_home'] > df['score_away']).astype(int)

    # Team strength: Elo and ranks
    # Ensure numeric
    df['homePregameElo'] = pd.to_numeric(df['homePregameElo'], errors='coerce')
    df['awayPregameElo'] = pd.to_numeric(df['awayPregameElo'], errors='coerce')

    # Elo difference: positive => home stronger
    df['elo_diff'] = df['homePregameElo'] - df['awayPregameElo']

    # Rank columns (lower is better)
    df['rank_home'] = pd.to_numeric(df['rank_home'], errors='coerce')
    df['rank_away'] = pd.to_numeric(df['rank_away'], errors='coerce')

    # Treat unranked as 26 (or whatever cutoff you like)
    df['rank_home_adj'] = df['rank_home'].fillna(26)
    df['rank_away_adj'] = df['rank_away'].fillna(26)

    # Rank difference: positive => home higher ranked (better)
    df['rank_diff'] = df['rank_away_adj'] - df['rank_home_adj']

    # Top-25 indicators
    df['home_top25'] = (df['rank_home'] <= 25).astype('Int64')
    df['away_top25'] = (df['rank_away'] <= 25).astype('Int64')

    df['home_top25_vs_not'] = (
        (df['home_top25'] == 1) & ((df['away_top25'] == 0) | df['away_top25'].isna())
    ).astype('Int64')

    df['away_top25_vs_not'] = (
        (df['away_top25'] == 1) & ((df['home_top25'] == 0) | df['home_top25'].isna())
    ).astype('Int64')

    # Venue / home-field context
    # Capacity
    df['capacity'] = pd.to_numeric(df['capacity'], errors='coerce')

    # Z-score capacity (for continuous effect)
    cap_mean = df['capacity'].mean(skipna=True)
    cap_std = df['capacity'].std(skipna=True)
    df['capacity_z'] = (df['capacity'] - cap_mean) / cap_std

    # Buckets for intuitive interpretation
    df['capacity_big'] = (df['capacity'] >= 80000).astype('Int64')
    df['capacity_med'] = ((df['capacity'] >= 50000) & (df['capacity'] < 80000)).astype('Int64')
    # small stadium is the implicit reference group

    # Elevation / altitude (assumes feet; adjust threshold if meters)
    df['elevation'] = pd.to_numeric(df['elevation'], errors='coerce')
    df['high_altitude'] = (df['elevation'] >= 3000).astype('Int64')

    # Dome / grass
    df['is_dome'] = df['dome'].fillna(False).astype(bool).astype('Int64')
    df['is_grass'] = df['grass'].fillna(False).astype(bool).astype('Int64')

    # Neutral site: combine schedule + CFBD flags
    neutral_sched = df['neutral'].fillna(0).astype(int)  # 1 if neutral, 0 otherwise
    neutral_cfbd = df['neutralSite'].fillna(False).astype(bool)
    df['is_neutral'] = ((neutral_sched == 1) | neutral_cfbd).astype('Int64')

    # Game timing (datetime, night game, week)
    # Try to build a single datetime in ET
    dt1 = pd.to_datetime(df['game_dt_et'], errors='coerce')
    dt2 = pd.to_datetime(
        df['date'].astype(str) + ' ' + df['time_et'].astype(str),
        errors='coerce'
    )
    df['game_datetime_et'] = dt1.fillna(dt2)

    # Hour of day in ET
    df['hour_et'] = df['game_datetime_et'].dt.hour
    df['night_game'] = (df['hour_et'] >= 18).astype('Int64')

    # Day of week (0=Mon,...,5=Sat,6=Sun)
    df['dow'] = df['game_datetime_et'].dt.dayofweek
    df['is_sat'] = (df['dow'] == 5).astype('Int64')
    df['is_sun'] = (df['dow'] == 6).astype('Int64')

    # Season / week normalization (for "how far into season" effect)
    df['season'] = pd.to_numeric(df['season'], errors='coerce')
    df['week'] = pd.to_numeric(df['week'], errors='coerce')

    max_week_by_season = df.groupby('season')['week'].transform('max')
    df['week_norm'] = df['week'] / max_week_by_season


    # TV / big stage

    # Big national networks; tweak list as you like
    big_tv_networks = ['ABC', 'FOX', 'CBS', 'NBC', 'ESPN', 'ESPN2', 'FOX Sports 1', 'FS1', 'CBS Sports Network']
    df['is_big_tv'] = df['tv'].isin(big_tv_networks).astype('Int64')


    # Conference game indicator as numeric

    df['conferenceGame_flag'] = df['conferenceGame'].fillna(False).astype(bool).astype('Int64')


    # Example interaction terms

    # Elo diff interacts with capacity and neutrality
    df['elo_diff_cap'] = df['elo_diff'] * df['capacity_z']
    df['elo_diff_non_neutral'] = df['elo_diff'] * (1 - df['is_neutral'].fillna(0))

    return df

# Example usage:
# df_features = add_home_away_features(df)
# Now df_features has all original columns + engineered ones.


In [ ]:
df = pd.read_csv("/Users/will/GitHub/IS477/data/cleaned/merged_games.csv", low_memory=False)

In [ ]:
add_home_away_features(df)

| Variable               | Description                                                                           | Calculated? (Source / Formula)                                                                                                |
| ---------------------- | ------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------- |
| `home_win`             | Binary target: home team win indicator.                                               | **Yes** – `1` if `score_home > score_away`, else `0`.                                                                         |
| `homePregameElo`       | Pregame Elo rating for the home team (higher = stronger).                             | **No** – raw column.                                                                                                          |
| `awayPregameElo`       | Pregame Elo rating for the away team (higher = stronger).                             | **No** – raw column.                                                                                                          |
| `rank_home`            | Poll ranking for the home team; lower values indicate better ranking; NaN ≈ unranked. | **No** – raw column.                                                                                                          |
| `rank_away`            | Poll ranking for the away team; lower values indicate better ranking; NaN ≈ unranked. | **No** – raw column.                                                                                                          |
| `elo_diff`             | Elo strength advantage for home team.                                                 | **Yes** – `homePregameElo - awayPregameElo`.                                                                                  |
| `rank_home_adj`        | Home team rank with unranked treated as worse than 25.                                | **Yes** – `rank_home` with NaN replaced by `26`.                                                                              |
| `rank_away_adj`        | Away team rank with unranked treated as worse than 25.                                | **Yes** – `rank_away` with NaN replaced by `26`.                                                                              |
| `rank_diff`            | Rank advantage for home team (top-25 style).                                          | **Yes** – `rank_away_adj - rank_home_adj` (positive ⇒ home is better ranked).                                                 |
| `home_top25`           | Indicator for home team being ranked in AP-style Top 25.                              | **Yes** – `1` if `rank_home <= 25`, else `0`/NA.                                                                              |
| `away_top25`           | Indicator for away team being ranked in AP-style Top 25.                              | **Yes** – `1` if `rank_away <= 25`, else `0`/NA.                                                                              |
| `home_top25_vs_not`    | Home is ranked, away is not.                                                          | **Yes** – `1` if `home_top25 == 1` and (`away_top25 == 0` or NA), else `0`.                                                   |
| `away_top25_vs_not`    | Away is ranked, home is not.                                                          | **Yes** – `1` if `away_top25 == 1` and (`home_top25 == 0` or NA), else `0`.                                                   |
| `capacity`             | Stadium seating capacity (nominal maximum attendance).                                | **No** – raw column (converted to numeric).                                                                                   |
| `capacity_z`           | Standardized stadium size (for continuous effect).                                    | **Yes** – `(capacity - mean(capacity)) / std(capacity)` over dataset.                                                         |
| `capacity_big`         | Indicator for very large stadium.                                                     | **Yes** – `1` if `capacity >= 80000`, else `0`.                                                                               |
| `capacity_med`         | Indicator for medium-size stadium.                                                    | **Yes** – `1` if `50000 <= capacity < 80000`, else `0`. (Small stadium is reference.)                                         |
| `elevation`            | Venue elevation (assumed feet unless otherwise noted).                                | **No** – raw column (converted to numeric).                                                                                   |
| `high_altitude`        | Indicator for high-elevation venues.                                                  | **Yes** – `1` if `elevation >= 3000`, else `0`.                                                                               |
| `dome`                 | Raw flag indicating if venue is a dome/enclosed.                                      | **No** – raw column.                                                                                                          |
| `grass`                | Raw flag indicating if surface is natural grass.                                      | **No** – raw column.                                                                                                          |
| `is_dome`              | Clean binary dome indicator.                                                          | **Yes** – from `dome` after filling missing and casting to `0/1`.                                                             |
| `is_grass`             | Clean binary grass indicator.                                                         | **Yes** – from `grass` after filling missing and casting to `0/1`.                                                            |
| `neutral`              | Schedule-level neutral-site flag (0/1).                                               | **No** – raw column.                                                                                                          |
| `neutralSite`          | CFBD neutral-site flag (bool/0/1).                                                    | **No** – raw column.                                                                                                          |
| `is_neutral`           | Combined neutral-site indicator.                                                      | **Yes** – `1` if `neutral == 1` **or** `neutralSite == True`, else `0`.                                                       |
| `timezone`             | Time zone of venue (e.g. `America/New_York`).                                         | **No** – raw column.                                                                                                          |
| `game_dt_et`           | Raw game datetime in ET from schedule.                                                | **No** – raw column.                                                                                                          |
| `date`                 | Game date (usually `YYYY-MM-DD`).                                                     | **No** – raw column.                                                                                                          |
| `time_et`              | Scheduled kickoff time in Eastern Time.                                               | **No** – raw column.                                                                                                          |
| `game_datetime_et`     | Parsed game datetime in ET used for timing features.                                  | **Yes** – `pd.to_datetime(game_dt_et)`; if NaT, fallback to `pd.to_datetime(date + ' ' + time_et)`.                           |
| `hour_et`              | Hour of day for kickoff in ET (0–23).                                                 | **Yes** – `game_datetime_et.dt.hour`.                                                                                         |
| `night_game`           | Indicator for night/prime-time game.                                                  | **Yes** – `1` if `hour_et >= 18`, else `0`.                                                                                   |
| `dow`                  | Day of week (0=Mon,…,5=Sat,6=Sun).                                                    | **Yes** – `game_datetime_et.dt.dayofweek`.                                                                                    |
| `is_sat`               | Indicator for Saturday games.                                                         | **Yes** – `1` if `dow == 5`, else `0`.                                                                                        |
| `is_sun`               | Indicator for Sunday games.                                                           | **Yes** – `1` if `dow == 6`, else `0`.                                                                                        |
| `season`               | Season year (e.g., 2023).                                                             | **No** – raw column (converted to numeric).                                                                                   |
| `week`                 | Week number within the season.                                                        | **No** – raw column (converted to numeric).                                                                                   |
| `week_norm`            | Where this game falls in the season.                                                  | **Yes** – `week / max(week)` **within each season** (via groupby transform).                                                  |
| `tv`                   | TV network / channel carrying the game.                                               | **No** – raw column.                                                                                                          |
| `is_big_tv`            | Indicator for national TV exposure.                                                   | **Yes** – `1` if `tv` ∈ {`ABC`, `FOX`, `CBS`, `NBC`, `ESPN`, `ESPN2`, `FOX Sports 1`, `FS1`, `CBS Sports Network`}, else `0`. |
| `homeConference`       | Conference of the home team (e.g., Big Ten).                                          | **No** – raw column.                                                                                                          |
| `awayConference`       | Conference of the away team.                                                          | **No** – raw column.                                                                                                          |
| `homeClassification`   | Home team classification (e.g., FBS, FCS).                                            | **No** – raw column.                                                                                                          |
| `awayClassification`   | Away team classification (e.g., FBS, FCS).                                            | **No** – raw column.                                                                                                          |
| `seasonType`           | Season type (e.g., regular, postseason).                                              | **No** – raw column.                                                                                                          |
| `conferenceGame`       | Raw flag for whether game is a conference game.                                       | **No** – raw column.                                                                                                          |
| `conferenceGame_flag`  | Clean numeric conference-game indicator.                                              | **Yes** – `1` if `conferenceGame == True`, else `0`.                                                                          |
| `elo_diff_cap`         | Interaction: Elo advantage × stadium size.                                            | **Yes** – `elo_diff * capacity_z`.                                                                                            |
| `elo_diff_non_neutral` | Interaction: Elo advantage × non-neutral status.                                      | **Yes** – `elo_diff * (1 - is_neutral)`.                                                                                      |


In [ ]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

In [ ]:
na_stats = (
    df.isna().sum()  # count NAs per column
      .to_frame(name='n_missing')
      .assign(
          n_rows=len(df),
          pct_missing=lambda s: (s['n_missing'] / s['n_rows']) * 100,
          n_non_missing=lambda s: s['n_rows'] - s['n_missing'],
          dtype=df.dtypes
      )
      .reset_index()
      .rename(columns={'index': 'column'})
      .sort_values('pct_missing', ascending=False)
)

print(na_stats)